In [1]:
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from torch import autograd
import time

# LeNet Implementation in PyTorch

This notebook implements the LeNet-5 convolutional neural network for handwritten digit recognition using MNIST dataset.

In [2]:
# Check for MPS (Apple Silicon GPU) availability
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.pool = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5, padding=2)
        self.conv3 = nn.Conv2d(16, 120, 5)
        self.pool3 = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, 10)
   
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool3(torch.relu(self.conv3(x)))
        x = x.view(-1, 120)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [4]:
import torchvision
import torchvision.transforms as transforms

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load MNIST dataset
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [5]:
# Initialize model, loss, and optimizer
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('Training finished')

Epoch 1, Batch 100, Loss: 2.298
Epoch 1, Batch 200, Loss: 2.249
Epoch 1, Batch 300, Loss: 1.436
Epoch 1, Batch 400, Loss: 0.718
Epoch 1, Batch 500, Loss: 0.420
Epoch 1, Batch 600, Loss: 0.276
Epoch 1, Batch 700, Loss: 0.234
Epoch 1, Batch 800, Loss: 0.202
Epoch 1, Batch 900, Loss: 0.180
Epoch 2, Batch 100, Loss: 0.144
Epoch 2, Batch 200, Loss: 0.138
Epoch 2, Batch 300, Loss: 0.130
Epoch 2, Batch 400, Loss: 0.138
Epoch 2, Batch 500, Loss: 0.112
Epoch 2, Batch 600, Loss: 0.110
Epoch 2, Batch 700, Loss: 0.111
Epoch 2, Batch 800, Loss: 0.115
Epoch 2, Batch 900, Loss: 0.102
Epoch 3, Batch 100, Loss: 0.091
Epoch 3, Batch 200, Loss: 0.093
Epoch 3, Batch 300, Loss: 0.086
Epoch 3, Batch 400, Loss: 0.091
Epoch 3, Batch 500, Loss: 0.077
Epoch 3, Batch 600, Loss: 0.084
Epoch 3, Batch 700, Loss: 0.084
Epoch 3, Batch 800, Loss: 0.082
Epoch 3, Batch 900, Loss: 0.074
Epoch 4, Batch 100, Loss: 0.071
Epoch 4, Batch 200, Loss: 0.073
Epoch 4, Batch 300, Loss: 0.066
Epoch 4, Batch 400, Loss: 0.075
Epoch 4,

In [6]:
# Move model to device
model = model.to(device)

# Test the model
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy on test set: {100 * correct / total:.2f}%')

Accuracy on test set: 98.46%


# AlexNet Implementation

This section implements AlexNet for MNIST digit recognition.

In [17]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), 256 * 6 * 6)
        x = self.classifier(x)
        return x

In [18]:
# Input preparation for AlexNet
alex_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

alex_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=alex_transform)
alex_trainloader = torch.utils.data.DataLoader(alex_trainset, batch_size=64, shuffle=True)

alex_testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=alex_transform)
alex_testloader = torch.utils.data.DataLoader(alex_testset, batch_size=64, shuffle=False)

In [19]:
# Train AlexNet
alex_model = AlexNet().to(device)
alex_criterion = nn.CrossEntropyLoss()
alex_optimizer = optim.SGD(alex_model.parameters(), lr=0.01, momentum=0.9)

alex_num_epochs = 5
for epoch in range(alex_num_epochs):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(alex_trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        alex_optimizer.zero_grad()
        outputs = alex_model(inputs)
        loss = alex_criterion(outputs, labels)
        loss.backward()
        alex_optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'AlexNet Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('AlexNet Training finished')

AlexNet Epoch 1, Batch 100, Loss: 2.301
AlexNet Epoch 1, Batch 200, Loss: 1.819
AlexNet Epoch 1, Batch 300, Loss: 0.328
AlexNet Epoch 1, Batch 400, Loss: 0.199
AlexNet Epoch 1, Batch 500, Loss: 0.124
AlexNet Epoch 1, Batch 600, Loss: 0.113
AlexNet Epoch 1, Batch 700, Loss: 0.096
AlexNet Epoch 1, Batch 800, Loss: 0.085
AlexNet Epoch 1, Batch 900, Loss: 0.068
AlexNet Epoch 2, Batch 100, Loss: 0.071
AlexNet Epoch 2, Batch 200, Loss: 0.067
AlexNet Epoch 2, Batch 300, Loss: 0.063
AlexNet Epoch 2, Batch 400, Loss: 0.072
AlexNet Epoch 2, Batch 500, Loss: 0.045
AlexNet Epoch 2, Batch 600, Loss: 0.049
AlexNet Epoch 2, Batch 700, Loss: 0.057
AlexNet Epoch 2, Batch 800, Loss: 0.062
AlexNet Epoch 2, Batch 900, Loss: 0.055
AlexNet Epoch 3, Batch 100, Loss: 0.037
AlexNet Epoch 3, Batch 200, Loss: 0.041
AlexNet Epoch 3, Batch 300, Loss: 0.038
AlexNet Epoch 3, Batch 400, Loss: 0.049
AlexNet Epoch 3, Batch 500, Loss: 0.041
AlexNet Epoch 3, Batch 600, Loss: 0.038
AlexNet Epoch 3, Batch 700, Loss: 0.036


In [20]:
# Test AlexNet
alex_correct = 0
alex_total = 0
with torch.no_grad():
    for inputs, labels in alex_testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = alex_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        alex_total += labels.size(0)
        alex_correct += (predicted == labels).sum().item()

print(f'AlexNet Accuracy on test set: {100 * alex_correct / alex_total:.2f}%')

AlexNet Accuracy on test set: 98.87%


In [32]:
# VGGNet (VGG-like) implementation for MNIST (resized to 224x224 RGB)
class VGGNet(nn.Module):
    def __init__(self, num_classes=10):
        super(VGGNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [31]:
%pip install torchsummary
from torchsummary import summary

VGG_model = VGGNet()
summary(VGG_model, (3, 224, 224))

Note: you may need to restart the kernel to use updated packages.
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 224, 224]           1,792
              ReLU-2         [-1, 64, 224, 224]               0
         MaxPool2d-3         [-1, 64, 112, 112]               0
            Conv2d-4        [-1, 128, 112, 112]          73,856
              ReLU-5        [-1, 128, 112, 112]               0
         MaxPool2d-6          [-1, 128, 56, 56]               0
            Conv2d-7          [-1, 256, 56, 56]         295,168
              ReLU-8          [-1, 256, 56, 56]               0
            Conv2d-9          [-1, 256, 56, 56]         590,080
             ReLU-10          [-1, 256, 56, 56]               0
        MaxPool2d-11          [-1, 256, 28, 28]               0
           Conv2d-12          [-1, 512, 28, 28]       1,180,160
             ReLU-13          [-1, 51

In [33]:
# VGGNet input transforms and data loaders (reuse alex_transform if present)
try:
    vgg_transform = alex_transform
except NameError:
    vgg_transform = transforms.Compose([
        transforms.Resize(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

vgg_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=vgg_transform)
vgg_trainloader = torch.utils.data.DataLoader(vgg_trainset, batch_size=64, shuffle=True)

vgg_testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=vgg_transform)
vgg_testloader = torch.utils.data.DataLoader(vgg_testset, batch_size=64, shuffle=False)

In [34]:
# Train VGGNet
vgg_model = VGGNet().to(device)
vgg_criterion = nn.CrossEntropyLoss()
vgg_optimizer = optim.SGD(vgg_model.parameters(), lr=0.01, momentum=0.9)

vgg_num_epochs = 3
for epoch in range(vgg_num_epochs):
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(vgg_trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        vgg_optimizer.zero_grad()
        outputs = vgg_model(inputs)
        loss = vgg_criterion(outputs, labels)
        loss.backward()
        vgg_optimizer.step()
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'VGGNet Epoch {epoch+1}, Batch {i+1}, Loss: {running_loss / 100:.3f}')
            running_loss = 0.0

print('VGGNet Training finished')

VGGNet Epoch 1, Batch 100, Loss: 2.302
VGGNet Epoch 1, Batch 200, Loss: 2.302
VGGNet Epoch 1, Batch 300, Loss: 2.303
VGGNet Epoch 1, Batch 400, Loss: 2.301
VGGNet Epoch 1, Batch 500, Loss: 2.301
VGGNet Epoch 1, Batch 600, Loss: 2.301
VGGNet Epoch 1, Batch 700, Loss: 2.302
VGGNet Epoch 1, Batch 800, Loss: 2.301
VGGNet Epoch 1, Batch 900, Loss: 2.301
VGGNet Epoch 2, Batch 100, Loss: 2.301
VGGNet Epoch 2, Batch 200, Loss: 2.301
VGGNet Epoch 2, Batch 300, Loss: 2.297
VGGNet Epoch 2, Batch 400, Loss: 2.283
VGGNet Epoch 2, Batch 500, Loss: 2.190
VGGNet Epoch 2, Batch 600, Loss: 0.608
VGGNet Epoch 2, Batch 700, Loss: 0.194
VGGNet Epoch 2, Batch 800, Loss: 0.145
VGGNet Epoch 2, Batch 900, Loss: 0.129
VGGNet Epoch 3, Batch 100, Loss: 0.092
VGGNet Epoch 3, Batch 200, Loss: 0.098
VGGNet Epoch 3, Batch 300, Loss: 0.089
VGGNet Epoch 3, Batch 400, Loss: 0.070
VGGNet Epoch 3, Batch 500, Loss: 0.078
VGGNet Epoch 3, Batch 600, Loss: 0.058
VGGNet Epoch 3, Batch 700, Loss: 0.064
VGGNet Epoch 3, Batch 800

In [ ]:
# Test VGGNet
vgg_correct = 0
vgg_total = 0
with torch.no_grad():
    for inputs, labels in vgg_testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = vgg_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        vgg_total += labels.size(0)
        vgg_correct += (predicted == labels).sum().item()

print(f'VGGNet Accuracy on test set: {100 * vgg_correct / vgg_total:.2f}%')

# ResNet-34 Implementation

This section implements ResNet-34 for MNIST digit recognition.